# 一筆解析パイプライン（team-alpha 版）— Phase 0

教材 `tutorials/04_field_pipeline.ipynb` のチーム用コピー。
**設定は `config.py`（再現性ヘッダ）から読み込む。このノート内に値をベタ打ちしない。**

教材からの主な変更点（理由は `config.py` と `analysis_doc.md` に記載）:

1. 再現性ヘッダを `config.py` の import に置換
2. 観測月を 4〜8月に（4月は湛水前ベースラインとして必須）
3. `CLOUDY_PIXEL_PERCENTAGE` によるシーン単位の足切りを**廃止**（SCL 画素単位マスクのみ）
4. 筆ポリゴンの `field_id` を `polygon_uuid` のまま保持（`field_001` 形式に振り直さない）
5. 画像取得範囲を ROI のベタ打ちから `fields.geometry().bounds()` に
6. フォールバック合成データの月マッピングのバグを修正
7. 出力先を `submissions/camp/team-alpha/` に
8. **SWIR（B11 / B12）由来の指標 `NDMI` / `NDTI` を追加**（`config.py` §3-2 の `S2_USE_SWIR`）。
   **チーム判断（9/5）により補助的な利用にとどめる**（`S2_EXTRA_INDEX_POLICY = 'auxiliary'`）。
   判定クラスと確信度は SWIR で変えない

**Phase 0 のスコープ**: 2024年・相馬市・田のみ・先頭 `N_LIMIT` 筆。
拡大手順は `analysis_doc.md` §5【段階的拡張の方針】を参照。


## GEE と Python の役割分担

「全部 GEE でやるの？全部 Python でやるの？」で混乱しないように、役割を分けます。

| 担当 | 仕事 |
|------|------|
| **GEE（クラウド）** | Sentinel-1/2 の取得・時間フィルタ・雲除去・軌道方向統一・バンド選択・NDVI計算・SAR前処理・**空間集計（Zonal Statistics）**・Export |
| **Python（Colab）** | DataFrame化・**波形特徴量の整理**・可視化・統計・閾値分析・機械学習・混同行列・F1/P/R/IoU・結果CSV/GIS生成 |

> **画像の重い処理は GEE に、表の加工と解析は Python に。** このノートもこの分担に沿って進みます。


---

## 学習目標
- **Zonal Statistics**（`reduceRegions`）で、農地ポリゴンごとに月別の NDVI / VV / VH 平均を集計できる
- 「単一の画像」ではなく「**時系列の波形**」から特徴量を作る考え方を身につける
  - 休耕地は「NDVIが低い」ではなく「**年間を通じて生育ピークが形成されない**」
- 最終課題で使える **`field_features.csv`（1筆×1年=1行）** を完成させる


In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install -q geemap earthengine-api pandas scikit-learn matplotlib japanize-matplotlib geopandas


In [ ]:
# ---- ライブラリ読み込み ----
import os, sys, glob
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 110

try:
    import japanize_matplotlib
except ImportError:
    japanize_matplotlib = None

# ---- Colab なら Google Drive をマウントする ----
# ランタイムが切れるとマウントは消える。開き直すたびに必要。
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive をマウントしました')

# ---- リポジトリ内の tutorials/ を import パスに通す ----
# 【教材との差異】教材は `from env import ...` を素で試し、失敗しても警告だけ出して
# 続行する。Colab の作業ディレクトリは /content なので必ず ImportError になり、
# GEE 未初期化のままセル7で落ちる。さらに satellite_utils も読めず、
# 雲マスクが一切かからないまま解析が進む（本設計では致命的）。
_CANDIDATES = [
    '/content/drive/MyDrive/astrocamp-2026-sia/tutorials',
    os.path.join(os.getcwd(), 'tutorials'),
    os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'tutorials')),
]
# 上で見つからなければ MyDrive 以下を2階層まで探す（クローン先が違っても拾う）
_CANDIDATES += [os.path.dirname(f) for f in
                glob.glob('/content/drive/MyDrive/*/tutorials/env.py') +
                glob.glob('/content/drive/MyDrive/*/*/tutorials/env.py')]

TUTORIALS_DIR = next((d for d in _CANDIDATES if os.path.exists(os.path.join(d, 'env.py'))), None)
if TUTORIALS_DIR is None:
    msg = ['env.py が見つかりません。探した場所:'] + ['  ' + d for d in _CANDIDATES]
    if os.path.exists('/content/drive/MyDrive'):
        msg.append('MyDrive の中身: ' + str(sorted(os.listdir('/content/drive/MyDrive'))[:30]))
    else:
        msg.append('/content/drive/MyDrive が存在しません。Drive のマウントに失敗しています。')
    raise FileNotFoundError('\n'.join(msg))

if TUTORIALS_DIR not in sys.path:
    sys.path.insert(0, TUTORIALS_DIR)
REPO_DIR = os.path.dirname(TUTORIALS_DIR)
print('repo     :', REPO_DIR)
print('tutorials:', TUTORIALS_DIR)

from env import gee_setup, setup_font, data_dir
import satellite_utils          # ここで失敗したら雲マスクが効かないので落とす
print('satellite_utils: OK')

# ---- GEE の認証・初期化 ----
# 未認証ならブラウザ認証が走る。プロジェクトIDは環境変数 GEE_PROJECT か対話入力。
gee_setup()

# 初期化できたことを実際の通信で確認する（ここを通れば以降のセルは動く）
print('GEE 疎通確認:', ee.Number(1).add(1).getInfo(), '（2 なら OK）')

setup_font()
DATA_DIR = data_dir()
print('DATA_DIR:', DATA_DIR)


## 再現性ヘッダー — 設定を1か所にまとめる

解析条件をコードの中に散らばせず、**この1ブロックに集約**します。
3週間後に自分（や他のメンバー）が同じ結果を再現できるようにするための仕組みです。

- 値を変えたときは、**このブロックの値も必ず書き換える**のがルールです
- 技術レポート §8「再現性」にはこのブロックをそのまま転記します
- 全教材で共通の設定です（`resources/product-related/guide_camp.md` §5 と同じ内容）


In [ ]:
# ============================================================
# 再現性ヘッダは config.py に集約している。ここでは読み込むだけ。
# 値を変えるときは config.py を編集すること（このセルに書かない）。
# ============================================================
import os, sys

# team-alpha ディレクトリを import パスに追加（Colab / ローカル両対応）
for _p in ['.', 'submissions/camp/team-alpha',
           '/content/drive/MyDrive/astrocamp-2026-sia/submissions/camp/team-alpha']:
    if os.path.exists(os.path.join(_p, 'config.py')) and _p not in sys.path:
        sys.path.insert(0, _p); break
# 【重要】config.py は必ずリロードする。
# Colab のランタイムは import 済みモジュールを sys.modules にキャッシュするため、
# git pull で config.py を更新してもランタイムを再起動しない限り古い値が返る
# （2026-09-08: N_OBS_MIN を 2 にしたのに None のまま扱われる事象が発生）。
import importlib
import config
importlib.reload(config)
cfg = config
print('config.py:', config.__file__)

# 教材セルが参照する名前へ割り当てる
START_DATE  = cfg.START_DATE
END_DATE    = cfg.END_DATE
YEARS       = cfg.PHASE0_YEARS      # Phase 0 は [2024]。Phase 1 で cfg.YEARS に戻す
MONTHS      = cfg.MONTHS            # [4,5,6,7,8]
TARGET_YEAR = cfg.TARGET_YEAR
CRS         = cfg.CRS
SCALE       = cfg.SCALE
S1_ORBIT    = cfg.S1_ORBIT
S1_MODE     = cfg.S1_MODE

print('=== 再現性ヘッダ（config.py） ===')
print('期間      :', START_DATE, '〜', END_DATE)
print('対象年    :', YEARS, '（TARGET_YEAR =', TARGET_YEAR, '）')
print('観測月    :', MONTHS)
print('CRS/scale :', CRS, '/', SCALE, 'm')
print('S1        :', S1_ORBIT, S1_MODE, '相対軌道', cfg.S1_RELATIVE_ORBIT)
print('S2 雲     : SCL 画素単位マスク', cfg.S2_SCL_MASK_CLASSES, '/ CPP 不使用 =', not cfg.S2_USE_CPP)
print('スコープ  : 市', cfg.CITY_CODES, '地目', cfg.LAND_TYPE, '/ 筆数上限', cfg.N_LIMIT)


In [ ]:
# 解析対象の範囲（config.py の ROI）= 筆の選定条件。Phase 0 は相馬市内 約7.2 km²
# 【注意】ROI は「どの筆を選んだか」の選定条件。画像取得範囲はセル9で
#         fields.geometry().bounds() に置き換える（筆の縁が ROI をはみ出すため）。
roi = ee.Geometry.Rectangle(cfg.ROI)
print('ROI（筆の選定条件）:', cfg.ROI)


## 圃場ポリゴン（農地区画）を準備する

ここで使う「一筆」とは、農地ポリゴンの**ひとつの区画**のことです。
本番では **農林水産省の「筆ポリゴン」**（公開データ）を読み込みます
（入手手順は `resources/data-guide.md`）。**未ダウンロードでも動くよう、サンプルにフォールバック**します。

- 実データ（Colab）: `/content/drive/MyDrive/astrocamp-2026/export/farmland/fukushima_polygons.geojson`
- 実データ（ローカル）: `tutorials/data/fukushima_polygons.geojson`（動作確認用サンプル）
- パスは環境変数 `ASTROCAMP_POLYGON_PATH` で上書きできます
- フォールバック: 02 と同じサンプル矩形（相馬・南相馬の農地付近）


In [ ]:
# ---- 圃場ポリゴンを読み込む（実ポリゴン → なければサンプル矩形） ----
import os
try:
    from env import polygon_path
    POLY_PATH = polygon_path()
except ImportError:
    POLY_PATH = '/content/drive/MyDrive/astrocamp-2026/export/farmland/fukushima_polygons.geojson'

SAMPLE_FIELDS = [
    ee.Feature(ee.Geometry.Rectangle([140.948, 37.787, 140.958, 37.797]), {'field_id': 'field_001'}),
    ee.Feature(ee.Geometry.Rectangle([140.960, 37.795, 140.968, 37.803]), {'field_id': 'field_002'}),
    ee.Feature(ee.Geometry.Rectangle([141.000, 37.636, 141.010, 37.644]), {'field_id': 'field_003'}),
    ee.Feature(ee.Geometry.Rectangle([140.996, 37.606, 141.006, 37.614]), {'field_id': 'field_004'}),
    ee.Feature(ee.Geometry.Rectangle([140.953, 37.782, 140.963, 37.792]), {'field_id': 'field_005'}),
    ee.Feature(ee.Geometry.Rectangle([140.966, 37.800, 140.974, 37.808]), {'field_id': 'field_006'}),
]

POLY_SOURCE = 'サンプル矩形（フォールバック）'
try:
    if not os.path.exists(POLY_PATH):
        raise FileNotFoundError(f'ポリゴンが見つかりません: {POLY_PATH}')
    import geopandas as gpd
    gdf = gpd.read_file(POLY_PATH)
    print(f'[ポリゴン] 読み込み {len(gdf)} 筆 / CRS {gdf.crs}')

    # スコープの絞り込み（ファイル側で絞ってあっても、二重に効かせて取り違えを防ぐ）
    if 'local_government_cd' in gdf.columns:
        gdf = gdf[gdf['local_government_cd'].isin(cfg.CITY_CODES)]
    if 'land_type' in gdf.columns:
        gdf = gdf[gdf['land_type'].isin(cfg.LAND_TYPE)]
    gdf = gdf.cx[cfg.ROI[0]:cfg.ROI[2], cfg.ROI[1]:cfg.ROI[3]]   # config の ROI に合わせる
    print(f'[ポリゴン] スコープ適用後 {len(gdf)} 筆'
          f'（市 {cfg.CITY_CODES} / 地目 {cfg.LAND_TYPE}）')

    if cfg.N_LIMIT is not None:
        gdf = gdf.head(cfg.N_LIMIT)
        print(f'[ポリゴン] N_LIMIT={cfg.N_LIMIT} で先頭を抽出')
    gdf = gdf.reset_index(drop=True)

    # 【教材との差異】field_id を 'field_000' 形式に振り直さない。
    # polygon_uuid は農水省データ側の識別子で、年度をまたいだ追跡や再実行時の
    # 突き合わせに使える。連番に置き換えると行の並び順に依存して意味が変わる。
    if 'field_id' not in gdf.columns:
        gdf['field_id'] = [f'field_{i:05d}' for i in range(len(gdf))]

    keep = [c for c in ['field_id', 'city', 'land_type', 'geometry'] if c in gdf.columns]
    fields = geemap.gdf_to_ee(gdf[keep])
    POLY_SOURCE = f'農水省筆ポリゴン（{cfg.TARGET_YEAR}年度版・実データ）'
except Exception as e:
    print(f'[フォールバック] サンプル矩形を使います: {e}')
    fields = ee.FeatureCollection(SAMPLE_FIELDS)

print('圃場ポリゴン数:', fields.size().getInfo(), '/ ソース:', POLY_SOURCE)
print('圃場 ID の例:', fields.limit(3).aggregate_array('field_id').getInfo())

# 【重要】画像取得の範囲を筆から導出する。ROI ちょうどでクリップすると、
# ROI 境界をまたぐ筆が部分的な範囲で集計され平均が偏る（旧 ROI 実測 71/8,581 = 0.83%）。
roi = fields.geometry().bounds()
print('画像取得範囲を fields.geometry().bounds() に更新しました')


## GEE でデータを準備する（S2 NDVI・S1 VV/VH）

03 と同じルールを守ります。
- Sentinel-2: `CLOUDY_PIXEL_PERCENTAGE < 20`（シーン単位）＋ SCL による画素単位の雲マスクの2段で雲を除去し、`normalizedDifference(['B8','B4'])` で NDVI
- Sentinel-1: IW モード・`orbitProperties_pass = DESCENDING` で**軌道方向を統一**（この地域は DESC しか通年で揃わない・下記）
- S1_GRD の VV/VH は**既に dB 値**なので、`10*log10` の二重変換をしない


In [ ]:
# ── Sentinel-2 の指標（NDVI・SWIR 系）と Sentinel-1 VV/VH（dB）のコレクションを準備 ──
def add_s2_indices(img):
    """S2 の1シーンに正規化指標バンドを付ける。

    指標は「シーンごとに計算してから月平均」する。先に反射率を月平均してから
    比を取ると、比の平均 ≠ 平均の比 でズレる（雲マスクで有効画素数が
    バンド間・日付間で違うため、ズレは無視できない）。
    """
    out = img.addBands(img.normalizedDifference(cfg.S2_NDVI_BANDS).rename('NDVI'))

    # SWIR（B11 / B12）由来の指標。config.py §3-2 を参照。
    #   NDMI = (B8 - B11)/(B8 + B11)   水分・湛水。水面と湿った土壌で上がる
    #   NDTI = (B11 - B12)/(B11 + B12) 残渣・耕起状態。S2 では NBR2 と同じ式
    # 生の B11 / B12 を列にしない理由: 両者は隣接 SWIR 帯域で相関が高く、
    # 増える情報は主に両者の差である（config.py 【多重共線性の注意】）。
    # 【解像度】B11 / B12 は 20 m。scale=10 で集計しても実質分解能は 20 m のまま。
    if getattr(cfg, 'S2_USE_SWIR', False):
        out = out.addBands([
            img.normalizedDifference(cfg.S2_NDMI_BANDS).rename('NDMI'),
            img.normalizedDifference(cfg.S2_NDTI_BANDS).rename('NDTI'),
        ])
    return out

# 【教材との差異】雲の除去は SCL の画素単位マスクのみ。
# 教材は ① CLOUDY_PIXEL_PERCENTAGE < 20（シーン単位）→ ② SCL（画素単位）の2段だが、
# チームは①を採用しない。CPP は MGRS タイル全体（110km四方）の値で、ROI（約415km²）は
# タイルの約29分の1にすぎず、雲量を測る範囲と判定を出す範囲が一致しないため。
# 付録D の実測: 偽通過 2024-08-26（CPP 2.62% だが ROI クリア率 27.1%）、
#               偽棄却 2024-04-13（CPP 20.82 だが ROI クリア率 98.1%）。
# 詳細は config.py の S2_USE_CPP を参照。
try:
    from satellite_utils import s2_cloud_mask
except ImportError:
    print('[注意] satellite_utils.py が無いため画素単位の雲マスクを省略します')
    print('       → SCL マスクなしでは NDVI に雲が混ざる。必ず配置すること。')
    s2_cloud_mask = lambda img: img

_s2_base = (ee.ImageCollection(cfg.S2_COLLECTION)
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE))
    # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  ← 意図的に不採用（上記）

s2_ndvi_all   = _s2_base.map(s2_cloud_mask).map(add_s2_indices)   # 雲マスク後（解析用）
s2_nomask_all = _s2_base.map(add_s2_indices)                      # 雲マスク前（分母用）
# 【重要】有効画素率の分母は「その画素を実際に覆った観測の数」でなければならない。
# ROI と交差したシーン数を分母にすると、MGRS タイルや軌道の違いでその画素を覆って
# いないシーンまで数に入り、比が構造的に小さくなる
# （2026-09-08: 全1,175筆が全月で棄却された不具合の原因）。

s1_all = (ee.ImageCollection(cfg.S1_COLLECTION)
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', S1_MODE))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('orbitProperties_pass', S1_ORBIT)))

# 相対軌道番号での固定（DESCENDING かつ 46 で単一入射角になる）
if cfg.S1_RELATIVE_ORBIT is not None:
    _s1_fixed = s1_all.filter(ee.Filter.eq('relativeOrbitNumber_start', cfg.S1_RELATIVE_ORBIT))
    if _s1_fixed.size().getInfo() > 0:
        s1_all = _s1_fixed
        print('S1 相対軌道', cfg.S1_RELATIVE_ORBIT, 'に固定しました')
    else:
        print('[警告] 相対軌道', cfg.S1_RELATIVE_ORBIT, 'の観測が0件。軌道固定をスキップします')

# S1_GRD の VV/VH は既に dB。二重変換しない（cfg.S1_APPLY_LOG10 = False）
s1_db_all = s1_all.map(lambda img: img.select(['VV', 'VH']).rename(['VV_db', 'VH_db']))

print('S2 scenes:', s2_ndvi_all.size().getInfo(), '/ S1 scenes:', s1_db_all.size().getInfo())
print('S1 軌道の内訳:', s1_all.aggregate_histogram('orbitProperties_pass').getInfo())
years  = YEARS
months = MONTHS

# SWIR 由来の指標だけに使う月（NDVI・VV・VH の months とは独立に変えられる）。
# 収穫後（当地の水稲は9〜10月）の裸地・残渣を見たい場合は config.py の
# SWIR_MONTHS を MONTHS + [9, 10] にする。現行は MONTHS と同じ 4〜8月であり、
# 「収穫後の田と休耕地の区別」という用途は現行の観測窓では成立しない。
swir_months = list(getattr(cfg, 'SWIR_MONTHS', months)) if getattr(cfg, 'S2_USE_SWIR', False) else []
print('SWIR 指標:', 'NDMI / NDTI' if swir_months else '不使用', '/ 対象月:', swir_months)


## Zonal Statistics — 月別の平均値を「筆ごと」に集計する

ここが今回の山場です。**「一筆の中の画素を全部平均して、筆に1つの値を持たせる」** のが
Zonal Statistics（空間集計）です。GEE では `reduceRegions` で一気に行います。

```python
img.reduceRegions(collection=fields, reducer=ee.Reducer.mean(),
                  scale=10, crs='EPSG:32654')
#      ↑ 対象画像       ↑ 一筆ポリゴン   ↑ 筆内の平均   ↑ 画素サイズ  ↑ 集計する座標系
#
# scale と crs は必ず明示する。省略すると画像側の既定値で集計され、
# 「どの解像度・どの座標系で集計したか」がコードから読み取れなくなる。
```

月ごとに「その月の NDVI / VH / VV の平均画像」を作り、一括で筆に平均値を落とします。


## ✅ Check 1

今、1つのセルは何を表していますか？

> **1つの圃場の1か月の観測値**（月別平均 NDVI / VH / VV）

`monthly` DataFrame の1行＝ `field_id` × `year` × 1か月の平均値です。
この時点ではまだ「特徴量」ではなく、生の月別観測値です。

## 🔁 小さく検証する・変更に強い設計（軽い実践と説明）

いきなり全200筆を実行せず、段階的に確認しながら進めます。

```text
5筆
  ↓ 正しく動く？ → OK
20筆
  ↓ 値が妥当？ → OK
50筆
  ↓ 問題ない？ → OK
200筆（本番）
```

### なぜ小さく試すのか？

- **計算時間の短縮**だけでなく、**問題を早く発見するため**です
- GEE の `reduceRegions` は大量筆だと数分かかり、エラー時の再実行が重い
- 少数筆で「列名・値域・NaNの有無」を確認してからスケールアップ

### 変更に強い設計 = モジュール化

処理を以下の5モジュールに分けて考えると、後で修正・拡張が容易です：

```text
[1. 衛星データ取得]  S2/S1 コレクション、フィルタ、雲除去、軌道統一
[2. 前処理]         NDVI計算、dB確認、バンド選択
[3. 空間集計]       reduceRegions（月別平均）→ 月別DataFrame
[4. 特徴量生成]     波形特徴量14種の計算 → field_features.csv
[5. 判定・出力]     ルール/ML適用 → 判別マップ・優先リスト・GeoPackage
```

この分割なら、
- 対象地域を変える → 【1】【3】の範囲だけ変更
- 別作物（大豆・麦など）を対象にする → 【4】の特徴量定義だけ追加
- 判定手法をルールからMLに変える → 【5】だけ差し替え

### 技術 → 現場価値への翻訳

| 技術的な設計 | 自治体へのメリット（現場価値） |
|------------|--------------------------|
| 段階的な小規模検証 | 本番導入前に問題を発見し、失敗リスクを下げる |
| モジュール化（取得/集計/特徴量/判定/出力） | 別地域・別作物への展開がしやすい、追加開発コストを抑える |
| 共通関数化（物理ガード等） | 単位ミス・軌道混在・CRSズレを自動検知、属人化を防ぐ |

In [ ]:
# ── GEE: 観測日ごとに筆単位で集計する（AGGREGATION = per_observation）──
# 【設計・2026-09-08】月平均に丸めない。S-1 は12日回帰で月2〜3観測しかなく、
# 月平均は状態の違う観測（乾田と湛水）を混ぜて信号を消す。実測がその実例:
#   04-06 -16.37 dB(乾田) / 04-18 -20.63 dB(湛水) / 04-30 -20.37 dB(湛水)
#   → 月平均 -19.12 dB は実在しない状態。基準値にすると落ち込みが 4.50 dB に縮む
#     （観測値で取れば 7.25 dB）。理由は config.py の AGGREGATION を参照。
#
# 列名に観測日をそのまま入れる（VH_20240406 など）。列名から日付が読めるため、
# 別途マッピングを持つ必要がなく、Step4-4 の「日付」も復元できる。
PER_OBS = getattr(cfg, 'AGGREGATION', 'monthly_mean') == 'per_observation'
print('集計の粒度:', '観測日ごと' if PER_OBS else '月平均')

BASE_MONTH = 4          # 湛水判定の基準月（乾田）。analysis_doc Step4-3
MO_LO, MO_HI = min(months), max(months)

def _window(col, year):
    """対象年の観測月（4〜8月）に絞る。"""
    return col.filterDate(ee.Date.fromYMD(year, MO_LO, 1),
                          ee.Date.fromYMD(year, MO_HI, 1).advance(1, 'month'))

def _dates_of(col):
    """コレクションに含まれる観測日（YYYYMMDD）の一覧。同日の複数シーンは1つに畳む。"""
    fc = col.map(lambda im: ee.Feature(None, {'d': im.date().format('YYYYMMdd')}))
    return sorted({f['properties']['d'] for f in fc.getInfo()['features']})

def _on_date(col, d):
    """その日の観測を1枚に合成する（同日に複数タイルがある S-2 用）。"""
    st = ee.Date.parse('YYYYMMdd', d)
    return col.filterDate(st, st.advance(1, 'day')).mosaic()

def monthly_image(collection, band, year, mo):
    """月平均画像（SWIR など補助指標用。判定には使わない）。"""
    start = ee.Date.fromYMD(year, mo, 1)
    return (collection.filterDate(start, start.advance(1, 'month'))
            .select(band).mean().rename(f'{band}_{mo:02d}'))

def stack_image(year):
    """その年の観測を1枚のマルチバンド画像に束ねる。"""
    imgs = []
    s1w, s2w = _window(s1_db_all, year), _window(s2_ndvi_all, year)

    if PER_OBS:
        d1 = _dates_of(s1w)
        for d in d1:
            im = _on_date(s1w, d)
            imgs.append(im.select('VH_db').rename(f'VH_{d}'))
            imgs.append(im.select('VV_db').rename(f'VV_{d}'))

        d2 = _dates_of(s2w)
        for d in d2:
            imgs.append(_on_date(s2w, d).select('NDVI').rename(f'NDVI_{d}'))

        print(f'    {year}年 観測日 S-1 {len(d1)}日 / S-2 {len(d2)}日'
              f'（バンド {len(d1)*2 + len(d2)} 本）')
        print(f'      S-1: {d1}')
    else:
        for b in ['NDVI']:
            for mo in months:
                imgs.append(monthly_image(s2_ndvi_all, b, year, mo))
        for b in ['VH_db', 'VV_db']:
            for mo in months:
                imgs.append(monthly_image(s1_db_all, b, year, mo))

    # SWIR 由来の指標は月平均のまま（判定に使わない補助指標・列数を抑える）
    for b in (['NDMI', 'NDTI'] if swir_months else []):
        for mo in swir_months:
            imgs.append(monthly_image(s2_ndvi_all, b, year, mo))

    return ee.Image.cat(imgs).float()

def reduce_fields(year):
    """筆を cfg.REDUCE_CHUNK 件ずつに分けて reduceRegions し、結果を結合して返す。

    一括の getInfo() は応答上限を超えて例外になり、教材のフォールバック
    （合成データ）に落ちる。分割して取得する。
    """
    img   = stack_image(year)
    n     = int(fields.size().getInfo())
    chunk = int(getattr(cfg, 'REDUCE_CHUNK', 200) or n)
    flist = fields.toList(n)
    feats = []
    for start in range(0, n, chunk):
        sub = ee.FeatureCollection(flist.slice(start, min(start + chunk, n)))
        got = img.reduceRegions(collection=sub, reducer=ee.Reducer.mean(),
                                scale=SCALE, crs=CRS, maxPixelsPerRegion=1e9)
        feats.extend(got.getInfo()['features'])
        print(f'    {year}年 {min(start + chunk, n):>5}/{n} 筆', end='\r')
    print(f'    {year}年 {n}/{n} 筆 取得完了      ')
    return feats

SOURCE = 'GEE 実データ（Sentinel-2 / Sentinel-1）'
try:
    recs = []
    for y in years:
        for f in reduce_fields(y):
            p = dict(f['properties'])
            p['year'] = y
            recs.append(p)
    monthly = pd.DataFrame(recs)

    # 観測日ごとの列を特定する（列名の末尾8桁が日付）
    def _date_cols(prefix):
        return sorted([c for c in monthly.columns
                       if c.startswith(prefix) and c[len(prefix):].isdigit()
                       and len(c[len(prefix):]) == 8])
    VH_COLS   = _date_cols('VH_')
    VV_COLS   = _date_cols('VV_')
    NDVI_COLS = _date_cols('NDVI_')
    swir_cols = [f'{b}_{m:02d}' for b in (['NDMI', 'NDTI'] if swir_months else [])
                 for m in swir_months]

    keep = ['field_id', 'year'] + VH_COLS + VV_COLS + NDVI_COLS + swir_cols
    monthly = monthly[[c for c in keep if c in monthly.columns]]
    for c in VH_COLS + VV_COLS + NDVI_COLS + swir_cols:
        monthly[c] = pd.to_numeric(monthly[c], errors='coerce')

    print()
    print(f'観測日ごとの列: VH {len(VH_COLS)} / VV {len(VV_COLS)} / NDVI {len(NDVI_COLS)}')

    # ---- Step4-2: 観測の採否 ----
    # 【2026-09-08】月平均をやめたため、有効画素率という概念自体が不要になった。
    # NDVI が NaN の観測日は「その筆はその日雲だった」という意味で、
    # 晴れた観測数は per_observation から直接数えられる。近似は不要。
    _ndvi_month = {c: int(c[-8:][4:6]) for c in NDVI_COLS}
    print()
    print('Step4-2 晴れた観測数（NDVI が取れた観測日の数）')
    for mo in months:
        cols = [c for c, m_ in _ndvi_month.items() if m_ == mo]
        if not cols:
            print('  %2d月: 観測日なし' % mo); continue
        cnt = monthly[cols].notna().sum(axis=1)
        monthly[f'nobs_{mo:02d}'] = cnt
        monthly[f'clear_ratio_{mo:02d}'] = cnt / len(cols)
        print('  %2d月: 観測日 %2d 日 / 晴れ 中央値 %.1f 日 (最小 %d, 最大 %d)'
              % (mo, len(cols), cnt.median(), cnt.min(), cnt.max()))

    # 判別に直接使う 5月・8月に観測がまったく無い筆は落とす
    _KEY_MONTHS = [5, 8]
    before = len(monthly)
    for mo in _KEY_MONTHS:
        if f'nobs_{mo:02d}' in monthly.columns:
            monthly = monthly[monthly[f'nobs_{mo:02d}'] > 0]
    # S-1 は雲の影響を受けないが、念のため基準月に観測がある行に限る
    _apr_vh = [c for c in VH_COLS if int(c[-8:][4:6]) == BASE_MONTH]
    if _apr_vh:
        monthly = monthly[monthly[_apr_vh].notna().any(axis=1)]
    monthly = monthly.reset_index(drop=True)
    print(f'\n有効な筆×年: {len(monthly)} 行（5月・8月に晴れ観測が1日以上・{before} 行から）')
    assert len(monthly) > 0
except Exception as e:
    # 実ポリゴンを読めているときは合成データに落とさない（取り違え防止）
    if 'gdf' in dir() and gdf is not None and len(gdf) > 0:
        raise RuntimeError(
            f'GEE からの抽出に失敗しました: {e}\n'
            f'実筆ポリゴン {len(gdf)} 筆を読み込めているため、合成データには落としません。\n'
            'よくある原因: 1回の getInfo が大きすぎる → config.py の REDUCE_CHUNK を'
            ' 100 などに小さくして再実行してください。') from e
    raise

print('\nデータソース:', SOURCE)
print('形状:', monthly.shape)
print('圃場数:', monthly['field_id'].nunique(), '/ 年数:', monthly['year'].nunique())
monthly.head()


## 波形特徴量を計算する（Python 側）—「値」ではなく「波形の形」を見る

休耕地は「NDVI が低い」だけでなく、**「年間を通じて生育ピークが形成されない」** のが特徴です。
だから「単一時期の値」ではなく、**時系列波形から計算する特徴量**が判別に効きます。

- **Sentinel-2（NDVI）**: 年間最大値・最小値・振幅・5月/8月の値・**ピークの月**・**生育勾配**
- **Sentinel-1（VH）**: 5月/8月の値・**湛水期の急落（年間最小）**・季節振幅・**時間差分（8月−5月）**
- **VV/VH 比**: 5月と8月の比（構造（土壌・湛水）の情報）
- **SWIR（B11 / B12）**: `NDMI`（水分・湛水）と `NDTI`（残渣・耕起）の 5月/8月・最大・最小・振幅

> **SWIR 由来の特徴量は補助的にのみ使う**（`config.py` の `S2_EXTRA_INDEX_POLICY = 'auxiliary'`）。
> **判定クラスと Step4-6 の確信度は SWIR で変えない。** 使ってよいのは、バグチェック
> （下の相関チェック・湛水の SAR / 光学 整合確認）、「要確認」筆の説明材料、現地確認の
> 優先順位づけの補助、欠測時の傍証としての提示までである。
> 判定ルールに投入するには、NDVI との重複度の実測と閾値の根拠づけが別途必要になる。


In [ ]:
# ── Python: 観測日ごとの時系列から特徴量を計算する ──
# 【設計・2026-09-08】月平均に丸めず観測値そのものから計算する。
# 月別平均は比較用に導出するが、判定に使う量（湛水の落ち込み・生育開始）は
# すべて観測日ベースで求める。理由は config.py の AGGREGATION を参照。

def _dcols(df, prefix, year=None, month=None):
    """列名末尾8桁が日付の列を、年・月で絞って日付順に返す。"""
    out = []
    for c in df.columns:
        tail = c[len(prefix):]
        if not (c.startswith(prefix) and len(tail) == 8 and tail.isdigit()):
            continue
        if year is not None and int(tail[:4]) != year:
            continue
        if month is not None and int(tail[4:6]) != month:
            continue
        out.append(c)
    return sorted(out, key=lambda c: c[-8:])

def _nanmean(df, cols):
    return df[cols].mean(axis=1) if cols else pd.Series(np.nan, index=df.index)

BASE_MONTH = 4
THRESH_DROP = cfg.VH_FLOOD_DROP_DB
THRESH_NDVI = cfg.NDVI_GROWTH_START

out_rows = []
for yr, g in monthly.groupby('year', sort=True):
    g = g.copy()
    vh_cols   = _dcols(g, 'VH_',   year=yr)
    vv_cols   = _dcols(g, 'VV_',   year=yr)
    ndvi_cols = _dcols(g, 'NDVI_', year=yr)
    if not vh_cols or not ndvi_cols:
        print(f'[警告] {yr}年: 観測日の列が見つからないためスキップ')
        continue
    print(f'{yr}年: VH {len(vh_cols)}日 / NDVI {len(ndvi_cols)}日')

    # ================= Step4-3: 湛水（4月の乾田を基準）=================
    apr_cols = _dcols(g, 'VH_', year=yr, month=BASE_MONTH)
    if not apr_cols:
        raise ValueError(f'{yr}年の{BASE_MONTH}月に S-1 観測がありません')

    _A = g[apr_cols].to_numpy(dtype=float)
    apr_dates = np.array([int(c[-8:]) for c in apr_cols])
    # 乾田 = 4月の観測で最も明るい日（湛水すると暗くなる）
    with np.errstate(invalid='ignore'):
        dry_pos = np.nanargmax(np.where(np.isnan(_A), -np.inf, _A), axis=1)
    g['vh_apr']       = _A[np.arange(len(_A)), dry_pos]
    g['vh_apr_date']  = apr_dates[dry_pos]
    g['vh_apr_range'] = np.nanmax(_A, axis=1) - np.nanmin(_A, axis=1)
    g['vh_apr_mean']  = _nanmean(g, apr_cols)      # 旧実装の値（比較用）

    # 乾田の日より後の観測での最大落ち込み
    _V   = g[vh_cols].to_numpy(dtype=float)
    _dts = np.array([int(c[-8:]) for c in vh_cols])
    after = _dts[None, :] > g['vh_apr_date'].to_numpy()[:, None]
    drops = np.where(after, g['vh_apr'].to_numpy()[:, None] - _V, np.nan)
    all_nan = np.all(np.isnan(drops), axis=1)
    with np.errstate(invalid='ignore'):
        dmax = np.where(all_nan, np.nan, np.nanmax(np.where(np.isnan(drops), -np.inf, drops), axis=1))
        dpos = np.nanargmax(np.where(np.isnan(drops), -np.inf, drops), axis=1)
    g['vh_drop_max']   = dmax
    g['vh_drop_date']  = np.where(all_nan, np.nan, _dts[dpos])
    g['vh_drop_month'] = pd.Series(g['vh_drop_date'], index=g.index).map(
        lambda v: int(str(int(v))[4:6]) if pd.notna(v) else pd.NA).astype('Int64')

    # 5月の落ち込み（5月の観測での最大落ち込み）
    may_cols = _dcols(g, 'VH_', year=yr, month=5)
    if may_cols:
        M = g[may_cols].to_numpy(dtype=float)
        g['vh_drop_may'] = g['vh_apr'].to_numpy() - np.nanmin(M, axis=1)
    else:
        g['vh_drop_may'] = np.nan

    g['flood_detected'] = (g['vh_drop_max'] >= THRESH_DROP).astype('boolean')
    g.loc[g['vh_apr'].isna() | g['vh_drop_max'].isna(), 'flood_detected'] = pd.NA

    # ================= Step4-4: 生育開始（実際の日付）=================
    # 【未確認事項 #24 の解決】月平均では日付を復元できなかったが、
    # 観測日ごとに持つことで「NDVI が閾値を超えた最初の観測日」が実際に出せる。
    _N   = g[ndvi_cols].to_numpy(dtype=float)
    _nd  = np.array([int(c[-8:]) for c in ndvi_cols])
    over = (_N >= THRESH_NDVI) & ~np.isnan(_N)
    any_over = over.any(axis=1)
    first = np.argmax(over, axis=1)
    g['ndvi_growth_start_date'] = np.where(any_over, _nd[first], np.nan)
    g['ndvi_first_month_over']  = pd.Series(g['ndvi_growth_start_date'], index=g.index).map(
        lambda v: int(str(int(v))[4:6]) if pd.notna(v) else pd.NA).astype('Int64')
    g['growth_detected'] = pd.Series(any_over, index=g.index).astype('boolean')
    g.loc[np.isnan(_N).all(axis=1), 'growth_detected'] = pd.NA

    # ================= NDVI 波形 =================
    g['ndvi_max'] = np.nanmax(_N, axis=1)
    g['ndvi_min'] = np.nanmin(_N, axis=1)
    g['ndvi_amplitude'] = g['ndvi_max'] - g['ndvi_min']
    with np.errstate(invalid='ignore'):
        peak = np.nanargmax(np.where(np.isnan(_N), -np.inf, _N), axis=1)
    g['ndvi_max_date']  = _nd[peak]
    g['ndvi_max_month'] = pd.Series(g['ndvi_max_date'], index=g.index).map(
        lambda v: int(str(int(v))[4:6])).astype(int)
    g['ndvi_peak_at_edge'] = g['ndvi_max_month'].isin([min(months), max(months)])

    # 月別平均（比較用・05 との互換のため名前を維持）
    for mo, name in [(5, 'may'), (8, 'aug')]:
        g[f'ndvi_{name}'] = _nanmean(g, _dcols(g, 'NDVI_', year=yr, month=mo))
        g[f'vh_{name}']   = _nanmean(g, _dcols(g, 'VH_',   year=yr, month=mo))
        g[f'vv_{name}']   = _nanmean(g, _dcols(g, 'VV_',   year=yr, month=mo))
    g['vv_apr'] = _nanmean(g, _dcols(g, 'VV_', year=yr, month=BASE_MONTH))

    # 5月からピーク月までの平均生育速度。ピークが5月以前なら 0
    _m2p = g['ndvi_max_month'] - 5
    g['growth_slope'] = np.where(_m2p >= 1,
                                 (g['ndvi_max'] - g['ndvi_may']) / _m2p.where(_m2p >= 1, 1), 0.0)

    # ================= VH / VV 波形 =================
    g['vh_min'] = np.nanmin(_V, axis=1)
    g['vh_amplitude'] = np.nanmax(_V, axis=1) - g['vh_min']
    g['vh_temporal_diff'] = g['vh_aug'] - g['vh_may']
    g['vv_drop_may'] = g['vv_apr'] - g['vv_may']
    # dB では比は差。割り算は物理的に無意味（VH が 0 付近で発散する）
    g['vv_vh_may'] = g['vv_may'] - g['vh_may']
    g['vv_vh_aug'] = g['vv_aug'] - g['vh_aug']

    # ================= 観測数（Step4-2 / 4-6）=================
    nobs_cols = [f'nobs_{mo:02d}' for mo in months if f'nobs_{mo:02d}' in g.columns]
    g['n_obs_total']   = g[nobs_cols].sum(axis=1) if nobs_cols else np.nan
    g['n_obs_min']     = g[nobs_cols].min(axis=1) if nobs_cols else np.nan
    g['n_months_used'] = (g[nobs_cols] > 0).sum(axis=1) if nobs_cols else np.nan

    out_rows.append(g)

m = pd.concat(out_rows, ignore_index=True)

FEATURES = ['ndvi_max', 'ndvi_min', 'ndvi_amplitude', 'ndvi_may', 'ndvi_aug',
            'ndvi_max_month', 'ndvi_max_date', 'ndvi_peak_at_edge', 'growth_slope',
            'ndvi_growth_start_date', 'ndvi_first_month_over', 'growth_detected',
            'vh_apr', 'vh_apr_date', 'vh_apr_range', 'vh_apr_mean',
            'vh_may', 'vh_aug', 'vh_min', 'vh_amplitude', 'vh_temporal_diff',
            'vh_drop_may', 'vh_drop_max', 'vh_drop_date', 'vh_drop_month', 'flood_detected',
            'vv_apr', 'vv_drop_may', 'vv_vh_may', 'vv_vh_aug',
            'n_obs_total', 'n_obs_min', 'n_months_used']

print()
print('乾田ベースライン（観測日ごとの最大＝最も乾いた日）')
print('  vh_apr 中央値      : %.2f dB' % m['vh_apr'].median())
print('  vh_apr_mean 中央値 : %.2f dB （旧実装の月平均）' % m['vh_apr_mean'].median())
print('  差                 : %.2f dB' % (m['vh_apr'].median() - m['vh_apr_mean'].median()))
print('  4月内の変動幅 中央値: %.2f dB' % m['vh_apr_range'].median())
print('  採用された乾田日の分布:')
for d, n in m['vh_apr_date'].value_counts().sort_index().items():
    print('    %s : %5d 筆 (%.1f%%)' % (int(d), n, 100*n/len(m)))
print()
print('Step4-3 湛水: 落ち込み中央値 %.2f dB / 検出 %d 筆 (%.1f%%)'
      % (m['vh_drop_max'].median(), int(m['flood_detected'].fillna(False).sum()),
         100*m['flood_detected'].fillna(False).mean()))
print('Step4-4 生育: 検出 %d 筆 (%.1f%%)'
      % (int(m['growth_detected'].fillna(False).sum()), 100*m['growth_detected'].fillna(False).mean()))

# ---- SWIR 波形（NDMI / NDTI）。月平均のまま。判定には投入しない ----
# 【用途の限定】cfg.S2_EXTRA_INDEX_POLICY == 'auxiliary'（チーム判断 2026-09-05）。
# 許す用途・許さない用途の定義は config.py §3-2 と analysis_notes.md 付録G。
SWIR_FEATURES = []
if swir_months:
    ndmi = [f'NDMI_{mo:02d}' for mo in swir_months if f'NDMI_{mo:02d}' in m.columns]
    ndti = [f'NDTI_{mo:02d}' for mo in swir_months if f'NDTI_{mo:02d}' in m.columns]
    if ndmi and ndti:
        m['ndmi_max'] = m[ndmi].max(axis=1); m['ndmi_min'] = m[ndmi].min(axis=1)
        m['ndmi_amplitude'] = m['ndmi_max'] - m['ndmi_min']
        m['ndti_max'] = m[ndti].max(axis=1); m['ndti_min'] = m[ndti].min(axis=1)
        SWIR_FEATURES += ['ndmi_max', 'ndmi_min', 'ndmi_amplitude', 'ndti_max', 'ndti_min']
        for mo, name in [(5, 'may'), (8, 'aug')]:
            if f'NDMI_{mo:02d}' in m.columns:
                m[f'ndmi_{name}'] = m[f'NDMI_{mo:02d}']; m[f'ndti_{name}'] = m[f'NDTI_{mo:02d}']
                SWIR_FEATURES += [f'ndmi_{name}', f'ndti_{name}']
        post = [mo for mo in swir_months if mo >= 9]
        if not post:
            print()
            print('[注意] SWIR_MONTHS に9月以降が無いため、収穫後の特徴量は作られない。')
            print('       「収穫後の裸地／休耕地の区別」は現行の観測窓では評価できない。')

FEATURES = FEATURES + SWIR_FEATURES
field_features = m[['field_id', 'year'] + [c for c in FEATURES if c in m.columns]].copy()
field_features.head(10)


## 追加指標の相関チェック（多重共線性の実測）

`NDMI` は `NDVI` と **B8 を共有**するため、構造的に相関する。相関が高い指標を
「別々の証拠」として扱うと、付録D で `CLOUDY_PIXEL_PERCENTAGE` と `SCL` について
指摘したのと同じ **「独立でない検証」** になり、Step4-6 の確信度が水増しされる。

ここで実測する。判断の目安（**慣行的な目安であり理論的な根拠を持つ値ではない**）:

| 実測 | 解釈 | 対応 |
|------|------|------|
| \|r\| ≥ 0.9 | ほぼ同じ情報 | 判定には片方だけ。もう一方はバグチェック用 |
| 0.7 ≤ \|r\| < 0.9 | 重複が大きい | 判定に両方使うなら理由を書く |
| \|r\| < 0.7 | 独立性がある | 追加の情報として意味がある |


In [ ]:
# ── 追加指標の相関チェック（多重共線性の実測）──
# 出元: config.py §3-2【多重共線性の注意】/ analysis_doc §3【追加指標の使用方針】
if SWIR_FEATURES:
    # 【前提】この相関はデータソースに依存する。フォールバックの合成データは
    # 作付/休耕/森林の3型を手で置いた値なので、相関は「置いた値」を測っているだけで
    # 実測の意味を持たない。判断に使うのは GEE 実データで走ったときの値だけ。
    if SOURCE != 'GEE 実データ（Sentinel-2 / Sentinel-1）':
        print('[警告] データソースが', SOURCE, 'のため、以下の相関は判断に使えない。')

    corr = field_features[FEATURES].corr()

    # 1) NDVI 系 × SWIR 系 の相関（B8 の共有による構造的な相関を見る）
    ndvi_side = [f for f in FEATURES if f.startswith(('ndvi', 'growth'))]
    pairs = []
    for a in ndvi_side:
        for b in SWIR_FEATURES:
            pairs.append((a, b, corr.loc[a, b]))
    # 2) SWIR 系どうし（NDMI と NDTI はどちらも B11 を使う）
    for i, a in enumerate(SWIR_FEATURES):
        for b in SWIR_FEATURES[i + 1:]:
            pairs.append((a, b, corr.loc[a, b]))

    rep = (pd.DataFrame(pairs, columns=['feature_a', 'feature_b', 'r'])
             .assign(abs_r=lambda d: d['r'].abs())
             .sort_values('abs_r', ascending=False)
             .reset_index(drop=True))
    print('相関の高い順（上位15組）  n =', len(field_features), '行')
    print(rep.head(15).to_string(index=False, float_format=lambda v: f'{v: .3f}'))

    hi = rep[rep['abs_r'] >= 0.9]
    if len(hi):
        print('\n[要判断] |r| >= 0.9 の組が', len(hi), '組ある。')
        print('  → 判定に使うのは片方だけにし、もう一方はバグチェック用に回す。')
    else:
        print('\n|r| >= 0.9 の組は無し。')

    # 3) バグチェック本来の使い方: SAR の湛水シグナルと 5月の NDMI が整合するか。
    #    水稲の5月は湛水 → SAR は急落（vh が低い）・光学は NDMI 上昇 のはず。
    #    ここが逆相関にならなければ、どちらかの前処理を疑う（符号・単位・月ズレ）。
    if 'ndmi_may' in field_features.columns:
        r_flood = field_features['vh_may'].corr(field_features['ndmi_may'])
        print(f'\n5月: vh_may と ndmi_may の相関 r = {r_flood: .3f}')
        print('  期待: 負（湛水した筆ほど VH が低く NDMI が高い）。')
        print('  正または 0 付近なら、湛水シグナル・雲マスク・月の割り当てのいずれかを疑う。')
else:
    print('SWIR 指標が無効（cfg.S2_USE_SWIR = False）のため相関チェックは省略。')


## ✅ Check 2

この行は最終的に何になりますか？

> **1つの圃場の特徴量ベクトル（1筆＝1行）**

`field_features` DataFrame の1行＝ `field_id` × `year` の組み合わせで、
波形特徴量（`ndvi_max`, `ndvi_amplitude`, `vh_min` など14種類。
`cfg.S2_USE_SWIR = True` なら SWIR 由来の `ndmi_*` / `ndti_*` が加わる）が並んでいます。
この表が W8 の判別・精度評価の入力になります。

## ✅ Check 3

自治体職員はこの CSV をどう使うでしょうか？

> **要確認圃場の抽出、現場巡回の優先順位付け、作付状況の把握**

- `ndvi_amplitude` が小さい圃場 → 休耕の疑い → 現地確認リストに入れる
- `vh_min` が湛水期に急落していない圃場 → 作付でない可能性 → 優先確認
- 確信度スコア（W6で学んだ `predict_proba` の考え方）を後から付与して、
  「グレーゾーン（要確認）」だけを職員に渡せば、巡回作業を大幅に削減できる

## 🎯 技術を現場価値に翻訳する（ミニワーク）

以下の3項目について、**非専門家（自治体職員）に伝わる言葉**で
メリットを1〜2文ずつ書いてみてください。

```python
# ここに記入してください（W7 Exercise Log の③として提出）

translation = {
    "大量処理（GEEで多数圃場を一括処理）":
        "これまで職員が1筆ずつ確認していた作業を、広域にまとめて処理し、"
        "確認が必要な圃場だけを絞り込めます。",
    "変更に強い設計（モジュール化）":
        "対象地域や作物が変わっても、モジュール単位で差し替えられるため、"
        "新規導入時の開発工数を大幅に削減できます。",
    "小規模検証・再実行可能な設計":
        "少数筆で動作確認してから全体展開するため、本番環境での予期せぬ"
        "エラーや計算ミスを未然に防げます。"
}
```

> **ヒント:** 「技術的な機能」ではなく「職員の作業時間・コスト・業務負担・拡張性」
> に結びつける言葉を選びましょう。これは合宿の自治体向け提案スライドで再利用します。

## 「1筆×1年=1行」の解析データセットを完成させる

これが W7 の成果物です。**1行 = 1筆（圃場）× 1年** の表に、判別に使える特徴量が並んでいます。
次の W8 ではこの表に判別ルール／機械学習を適用し、目視検証セットで精度を測ります。


In [ ]:
# ── 解析データセットの保存（W8・合宿で読み込む） ──
import os, json

OUT_DIR = next((d for d in ['.', 'submissions/camp/team-alpha',
                            '/content/drive/MyDrive/astrocamp-2026-sia/submissions/camp/team-alpha']
                if os.path.exists(os.path.join(d, 'config.py'))), '.')

print('サンプル数（筆×年）:', len(field_features),
      '/ 圃場数:', field_features['field_id'].nunique())
print('特徴量:', FEATURES)

csv_path = os.path.join(OUT_DIR, 'field_features.csv')
field_features.to_csv(csv_path, index=False)
print('保存:', csv_path)

geo_path = os.path.join(OUT_DIR, 'fields.geojson')
# 【教材との差異・重要】教材は fields.getInfo() で GEE から取り直して保存するが、
# 筆数が増えると getInfo のペイロード上限に当たって例外になり、except で
# 「スキップ」されて**前回の古い fields.geojson が残る**。そのまま 05 を回すと
# field_id が field_features.csv と一致せず結合が 0 件になる（2026-09-08 に発生）。
# ジオメトリはローカルの gdf に既にあるので GEE に問い合わせる必要がない。
# ローカルから保存し、失敗した場合も古いファイルを残さず必ず気づけるようにする。
try:
    if 'gdf' in dir() and gdf is not None and len(gdf) > 0:
        _g = gdf[[c for c in ['field_id', 'city', 'land_type', 'geometry'] if c in gdf.columns]]
        _g.to_crs('EPSG:4326').to_file(geo_path, driver='GeoJSON')
        print('保存:', geo_path, f'（筆ジオメトリ {len(_g)} 筆・ローカルの gdf から）')
    else:
        with open(geo_path, 'w') as f:      # フォールバック経路（サンプル矩形）用
            json.dump(fields.getInfo(), f)
        print('保存:', geo_path, '（筆ジオメトリ・GEE から取得）')

    # field_features.csv と件数・ID が一致することを確認する。
    # ここで気づけないと 05 で結合0件になる。
    import geopandas as _gpd
    _chk = _gpd.read_file(geo_path)
    _common = set(_chk['field_id'].astype(str)) & set(field_features['field_id'].astype(str))
    print(f'  照合: geojson {len(_chk)} 筆 / CSV {field_features["field_id"].nunique()} 筆'
          f' / 共通 {len(_common)} 筆')
    if len(_common) == 0:
        raise RuntimeError('fields.geojson と field_features.csv の field_id が一致しません')
except Exception as e:
    if os.path.exists(geo_path):
        os.rename(geo_path, geo_path + '.stale')
        print(f'  古い {geo_path} を .stale に退避しました（誤って使わないため）')
    raise RuntimeError(f'fields.geojson の保存に失敗しました: {e}') from e

# ---- 品質チェック（合宿前チェックリスト 必須項目4 の完了条件）----
print('\n=== 品質チェック ===')
ok = True
_n_expected = cfg.N_LIMIT if cfg.N_LIMIT else None
checks = [
    ('1筆×1年=1行になっている',
     len(field_features) == len(field_features[['field_id', 'year']].drop_duplicates())),
    ('対象年 %d が含まれている' % cfg.TARGET_YEAR,
     cfg.TARGET_YEAR in set(field_features['year'])),
    ('5月・8月の特徴量が入っている',
     all(c in field_features.columns for c in ['vh_may', 'ndvi_aug'])),
    ('全行が NaN でない特徴量を持つ',
     bool(field_features[FEATURES].notna().any(axis=1).all())),
    ('SWIR 指標の設定と列が一致している',
     bool(swir_months) == any(c.startswith('ndmi_') for c in field_features.columns)),
]
for name, res in checks:
    print(('  OK   ' if res else '  NG   ') + name)
    ok &= bool(res)
print('総合:', 'PASS' if ok else 'FAIL —— 上の NG を確認すること')


## 途中結果の可視化 — 波形の「形」を見る

作付の筆は夏に**明確なピーク**を持つ、休耕の筆は**平坦**、という違いをプロットで確認します。
「値ではなく波形の形を見る」感覚をここで掴みます。


In [ ]:
# ── 代表的な筆の時系列を観測日ごとに描く ──
# 【2026-09-08 書き換え】月平均をやめたため、月の折れ線ではなく実際の観測日で描く。
# 乾田ベースライン（vh_apr_date）と最大落ち込み（vh_drop_date）を図上に示す。
import matplotlib.dates as mdates
from datetime import datetime

# 【重要】保存先はモジュール（セル）のスコープで解決する。
# 関数の中で dir() を使うとローカル名しか返らず OUT_DIR が見えないため、
# 図が /content に保存されて Drive に現れなかった（2026-09-08）。
_OUT = OUT_DIR if 'OUT_DIR' in globals() else '.'
print('図の保存先:', _OUT)

def _to_dt(yyyymmdd):
    return datetime.strptime(str(int(yyyymmdd)), '%Y%m%d')

def plot_series(fids, yr=None):
    yr = yr or cfg.TARGET_YEAR
    vcols = _dcols(monthly, 'VH_',   year=yr)
    ncols = _dcols(monthly, 'NDVI_', year=yr)
    vdt = [_to_dt(c[-8:]) for c in vcols]
    ndt = [_to_dt(c[-8:]) for c in ncols]

    fig, axes = plt.subplots(len(fids), 1, figsize=(10, 2.8 * len(fids)), sharex=True)
    if len(fids) == 1:
        axes = [axes]
    for ax, fid in zip(axes, fids):
        row = monthly[(monthly['field_id'] == fid) & (monthly['year'] == yr)]
        ft = field_features[(field_features['field_id'] == fid) &
                            (field_features['year'] == yr)]
        if row.empty:
            ax.set_title(str(fid) + ': データなし')
            continue
        r = row.iloc[0]
        f = ft.iloc[0] if len(ft) else None

        ax.plot(vdt, [r[c] for c in vcols], marker='o', color='tab:blue')
        ax.set_ylabel('VH (dB)', color='tab:blue')
        ax.tick_params(axis='y', labelcolor='tab:blue')

        ax2 = ax.twinx()
        ax2.plot(ndt, [r[c] for c in ncols], marker='s', color='tab:green', alpha=0.8)
        ax2.set_ylabel('NDVI', color='tab:green')
        ax2.tick_params(axis='y', labelcolor='tab:green')
        ax2.set_ylim(0, 1)
        ax2.axhline(cfg.NDVI_GROWTH_START, color='tab:green', ls=':', lw=1)

        title = str(fid)[:8]
        if f is not None:
            if pd.notna(f['vh_apr_date']):
                ax.axvline(_to_dt(f['vh_apr_date']), color='tab:orange', ls='--', lw=1.5)
                ax.annotate('乾田', (_to_dt(f['vh_apr_date']), f['vh_apr']),
                            textcoords='offset points', xytext=(4, 6),
                            fontsize=8, color='tab:orange')
            if pd.notna(f['vh_drop_date']):
                ax.axvline(_to_dt(f['vh_drop_date']), color='crimson', ls='--', lw=1.5)
            if pd.notna(f['vh_drop_max']):
                fl = f['flood_detected']
                mark = 'マル' if fl is True else ('バツ' if fl is False else '判定不可')
                mark = {'マル': '検出', 'バツ': '非検出'}.get(mark, mark)
                title += '  湛水%s（落ち込み %.1f dB）' % (mark, f['vh_drop_max'])
            else:
                title += '  湛水判定不可'
            if pd.notna(f['ndvi_growth_start_date']):
                title += ' / 生育開始 ' + str(int(f['ndvi_growth_start_date']))
        ax.set_title(title, fontsize=10)
        ax.grid(alpha=0.3)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    plt.tight_layout()
    plt.savefig(os.path.join(_OUT, 'diag_series.png'), dpi=150, bbox_inches='tight')
    print('保存:', os.path.join(_OUT, 'diag_series.png'))
    plt.show()

# 落ち込みの大小で3筆を選び、見比べる
_ff = field_features[field_features['year'] == cfg.TARGET_YEAR]
_hi = _ff[_ff['flood_detected'] == True].sort_values('vh_drop_max', ascending=False)
_lo = _ff[_ff['flood_detected'] == False].sort_values('vh_drop_max')
_picks = []
if len(_hi):
    _picks.append(_hi.iloc[0]['field_id'])
if len(_hi) > 1:
    _picks.append(_hi.iloc[len(_hi) // 2]['field_id'])
if len(_lo):
    _picks.append(_lo.iloc[len(_lo) // 2]['field_id'])
if not _picks:
    _picks = list(_ff['field_id'].head(3))
plot_series(_picks)

print('考察のヒント:')
print(' ・水稲: 4月上旬に明るく（乾田）→ 5月に急落（湛水）→ 7〜8月に戻る（繁茂）')
print(' ・4月内でも 04-18 以降は既に暗い場合がある（代かき）。乾田は最も明るい観測日')
print(' ・NDVI が緑の点線（%.1f）を超えた最初の日が「推定生育開始日」' % cfg.NDVI_GROWTH_START)
print(' ・湛水 非検出 かつ NDVI 上昇あり → 要確認（転作やベースライン欠落の可能性）')


---

## まとめ（W8 への接続）

- **Zonal Statistics（`reduceRegions`）** で「画像」を「**1筆×1年=1行の表**」に変換しました
- **波形特徴量**（振幅・ピークの月・生育勾配・VHの季節振幅・VV/VH比）を作りました
  - キーワードは「**値ではなく波形の形**」「休耕地は生育ピークが形成されない」
- 保存物: `field_features.csv`（特徴量表）／`fields.geojson`（筆ジオメトリ）

> **問い: この `field_features.csv` は本当に信頼できるのか？**

**次の W8:** この表に判別ルールまたは機械学習を適用し、
「疑似ラベル」と「目視検証セット」の違いを踏まえて **F1 / Precision / Recall / IoU** で本当の精度を測ります。
**次の W8:** 閾値感度分析で「なぜこの閾値か」を根拠づけます。
**次の W8:** 判別結果を地図（緑=作付 / 赤=休耕 / 灰=要確認）と「要確認優先リスト」にして自治体に説明します。

### W7 の提出物（ミニマム3点・提出先：チームの Drive または camp-* ブランチ・任意）

1. **`field_features.csv`** を作成する
2. **波形特徴量の確認メモ（3点）** を書く
   - 実際に使った特徴量を3つ選び、「何を表すか」「作付/休耕とどう関係するか」「実データでどう見えたか」
3. **技術→現場価値の翻訳（3行）** を書く
   - 上記ミニワークの3項目（大量処理・モジュール化・小規模検証）の翻訳文

※ `validation_labels.csv`・精度評価・閾値感度分析・自治体向けスライドは W8 で扱います。

## 診断: 4月のベースラインは乾田か（未確認事項 #8 / #3）

Phase 0 の実行で **4月の VH 中央値が −19.51 dB、5月との差が 2.28 dB しかない**ことが判明した。
裸地の VH は通常 −15〜−18 dB 程度なので、**4月の観測に湛水後のものが混ざっている疑い**がある。

計画書は田植えを5月上旬〜中旬としているが、**代かきは田植えの1〜2週間前に始まる。**
4月の推定観測日 04-06 / 04-18 / 04-30 のうち **04-30 は既に湛水している可能性がある。**
3観測を月平均するとベースラインが押し下げられ、「4月比の落ち込み」が縮む。

下のセルで**観測日ごとの VH** を出し、どの日から暗くなるかを直接確認する。
あわせて S-1 の実際の観測日が確定するため、**未確認事項 #8 も解決する。**

In [ ]:
# ── 診断: S-1 の観測日ごとの VH（未確認事項 #8 / #3）──
DIAG_YEAR    = cfg.TARGET_YEAR
DIAG_SAMPLE  = 100          # 代表として使う筆数（全筆だと geometry が重い）

s1_diag = s1_db_all.filterDate(f'{DIAG_YEAR}-03-01', f'{DIAG_YEAR}-07-01')
print('対象期間 3〜6月の S-1 観測数:', s1_diag.size().getInfo())

# 代表筆の集合体で日ごとの中央値を取る
_sample = ee.FeatureCollection(fields.toList(DIAG_SAMPLE))
_geom   = _sample.geometry()

def _per_date(img):
    return ee.Feature(None, {
        'date':  img.date().format('YYYY-MM-dd'),
        'orbit': img.get('relativeOrbitNumber_start'),
        'vh':    img.select('VH_db').reduceRegion(
                     ee.Reducer.median(), _geom, SCALE, maxPixels=1e9).get('VH_db'),
        'vv':    img.select('VV_db').reduceRegion(
                     ee.Reducer.median(), _geom, SCALE, maxPixels=1e9).get('VV_db'),
    })

rows = [f['properties'] for f in s1_diag.map(_per_date).getInfo()['features']]
diag = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
diag['vh'] = pd.to_numeric(diag['vh']); diag['vv'] = pd.to_numeric(diag['vv'])
diag['month'] = pd.to_datetime(diag['date']).dt.month

print(f'\n代表 {DIAG_SAMPLE} 筆の観測日ごとの中央値（{DIAG_YEAR}年）')
print('  date        orbit    VH(dB)   VV(dB)')
for _, r in diag.iterrows():
    mark = '  ← 4月' if r['month'] == 4 else ('  ← 5月' if r['month'] == 5 else '')
    print('  %s  %5s  %8.2f %8.2f%s' % (r['date'], r['orbit'], r['vh'], r['vv'], mark))

apr = diag[diag['month'] == 4]
may = diag[diag['month'] == 5]
print('\n=== 4月の観測 ===')
print('  日付   :', list(apr['date']))
print('  VH     :', [round(v, 2) for v in apr['vh']])
if len(apr) >= 2:
    print('  4月内の変動幅: %.2f dB（最明 %.2f → 最暗 %.2f）'
          % (apr['vh'].max() - apr['vh'].min(), apr['vh'].max(), apr['vh'].min()))
    print('  4月の月平均   : %.2f dB' % apr['vh'].mean())
    print('  4月の最明値   : %.2f dB  ← 乾田に最も近い状態' % apr['vh'].max())
print('\n=== 5月の観測 ===')
print('  VH     :', [round(v, 2) for v in may['vh']])
if len(apr) and len(may):
    print('\n【比較】')
    print('  月平均ベース  : 4月 %.2f − 5月最小 %.2f = %.2f dB'
          % (apr['vh'].mean(), may['vh'].min(), apr['vh'].mean() - may['vh'].min()))
    print('  4月最明ベース : 4月 %.2f − 5月最小 %.2f = %.2f dB'
          % (apr['vh'].max(), may['vh'].min(), apr['vh'].max() - may['vh'].min()))
    print('  → 後者が大きく、かつ閾値 %.1f dB を超えるなら、'
          'ベースラインの取り方が原因と判断できる' % cfg.VH_FLOOD_DROP_DB)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(pd.to_datetime(diag['date']), diag['vh'], marker='o', label='VH (dB)')
ax.plot(pd.to_datetime(diag['date']), diag['vv'], marker='s', alpha=0.6, label='VV (dB)')
for m, c in [(4, 'tab:green'), (5, 'tab:blue')]:
    sub = diag[diag['month'] == m]
    if len(sub):
        ax.axvspan(pd.to_datetime(sub['date']).min(), pd.to_datetime(sub['date']).max(),
                   alpha=0.12, color=c, label=f'{m}月')
ax.set_ylabel('後方散乱 (dB)'); ax.set_title(f'S-1 観測日ごとの後方散乱（代表{DIAG_SAMPLE}筆の中央値・{DIAG_YEAR}年）')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
_OUT_D = OUT_DIR if 'OUT_DIR' in globals() else '.'
plt.savefig(os.path.join(_OUT_D, 'diag_s1_dates.png'), dpi=150, bbox_inches='tight')
print('保存:', os.path.join(_OUT_D, 'diag_s1_dates.png'))
plt.show()

print('\n【読み方】')
print(' ・4月の中で後半の観測だけ暗い → 代かきが4月下旬に始まっている。')
print('   ベースラインを「4月の月平均」から「4月前半のみ」または「4月の最明値」に変えるべき')
print(' ・4月の3観測が同程度に暗い     → 4月時点で既に湛水しているか、この地域の乾田が暗い')
print(' ・VV も同時に下がっている       → 水面の鏡面反射。湛水として整合的')